In [25]:
import torch
import pandas as pd
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from sklearn.model_selection import StratifiedKFold

from utils_graph import load_nodes, build_edge_index
from gip_utils import compute_gip_edges
from features import compute_node_features

def normalize_node(node):

    node = str(node).strip().lower()

    return node

In [26]:
ROOT = Path().resolve().parents[1]

DATA = ROOT / "data" / "data_cleaned"
GRAPH_DIR = ROOT / "graphs"
GRAPH_DIR.mkdir(exist_ok=True)

In [27]:
print("Loading node mappings")

circ_map, n_circ = load_nodes(DATA / "circRNA_nodes_clean.csv", "circRNA")
mir_map, n_mir = load_nodes(DATA / "miRNA_nodes_clean.csv", "miRNA")
dis_map, n_dis = load_nodes(DATA / "disease_nodes_clean.csv", "disease")

print("circRNA:", n_circ)
print("miRNA:", n_mir)
print("disease:", n_dis)

features = compute_node_features()

Loading node mappings
circRNA: 828
miRNA: 682
disease: 122
Graph nodes: 1471
Graph edges: 2709


In [28]:
print("Loading datasets")
pos_cd = pd.read_csv(DATA / "circRNA_disease_edges.csv")
neg_cd = pd.read_csv(DATA / "circRNA_disease_neg_edges.csv")

pos_cd['label'] = 1
neg_cd['label'] = 0

cd_edges = pd.concat([pos_cd, neg_cd], ignore_index=True)
cd_edges = cd_edges.sample(frac=1, random_state=42).reset_index(drop=True)
cm_edges = pd.read_csv(DATA / "circRNA_miRNA_edges.csv")
md_edges = pd.read_csv(DATA / "miRNA_disease_edges.csv")

print(cd_edges.shape, cm_edges.shape, md_edges.shape)

Loading datasets
(1970, 3) (896, 2) (828, 2)


In [29]:
print("Building circRNA–miRNA graph")

cm_src = torch.tensor(cm_edges["circRNA"].map(circ_map).values, dtype=torch.long)
cm_dst = torch.tensor(cm_edges["miRNA"].map(mir_map).values, dtype=torch.long)

cm_edge = build_edge_index(
    cm_edges,
    circ_map,
    mir_map,
    "circRNA",
    "miRNA"
)

# SHIFT miRNA node indices
mir_offset = n_circ
cm_edge[1] += mir_offset

circ_cm_gip, _, mir_cm_gip, _ = compute_gip_edges(
    cm_src,
    cm_dst,
    n_circ,
    n_mir,
    k=10
)

# SHIFT miRNA GIP edges
mir_cm_gip += mir_offset

edge_index_cm = torch.cat([
    cm_edge,
    circ_cm_gip,
    mir_cm_gip
], dim=1)

edge_index_cm = to_undirected(edge_index_cm)

# ------------------------
# Attach features
# ------------------------

x_cm = []

circ_nodes = sorted(circ_map.keys(), key=lambda x: circ_map[x])
mir_nodes = sorted(mir_map.keys(), key=lambda x: mir_map[x])

for node in circ_nodes:
    x_cm.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

for node in mir_nodes:
    x_cm.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

x_cm = torch.tensor(x_cm, dtype=torch.float)

gcm = Data()
gcm.edge_index = edge_index_cm
gcm.x = x_cm
gcm.num_circ = n_circ
gcm.num_mir = n_mir

torch.save(gcm, GRAPH_DIR / "gcm_graph.pt")

print("Saved gcm_graph.pt")
print("Feature shape:", gcm.x.shape)

Building circRNA–miRNA graph
Saved gcm_graph.pt
Feature shape: torch.Size([1510, 6])


In [30]:
print("Building miRNA–disease graph")

md_src = torch.tensor(md_edges["miRNA"].map(mir_map).values, dtype=torch.long)
md_dst = torch.tensor(md_edges["disease"].map(dis_map).values, dtype=torch.long)

md_edge = build_edge_index(
    md_edges,
    mir_map,
    dis_map,
    "miRNA",
    "disease"
)

# SHIFT disease node indices
dis_offset = n_mir
md_edge[1] += dis_offset

mir_md_gip, _, dis_md_gip, _ = compute_gip_edges(
    md_src,
    md_dst,
    n_mir,
    n_dis,
    k=10
)

# SHIFT disease GIP edges
dis_md_gip += dis_offset

edge_index_md = torch.cat([
    md_edge,
    mir_md_gip,
    dis_md_gip
], dim=1)

edge_index_md = to_undirected(edge_index_md)

# ------------------------
# Attach features
# ------------------------

x_md = []

mir_nodes = sorted(mir_map.keys(), key=lambda x: mir_map[x])
dis_nodes = sorted(dis_map.keys(), key=lambda x: dis_map[x])

for node in mir_nodes:
    x_md.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

for node in dis_nodes:
    x_md.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

x_md = torch.tensor(x_md, dtype=torch.float)

gmd = Data()
gmd.edge_index = edge_index_md
gmd.x = x_md
gmd.num_mir = n_mir
gmd.num_dis = n_dis

torch.save(gmd, GRAPH_DIR / "gmd_graph.pt")

print("Saved gmd_graph.pt")
print("Feature shape:", gmd.x.shape)

Building miRNA–disease graph
Saved gmd_graph.pt
Feature shape: torch.Size([804, 6])


In [31]:
print("Creating 5-fold splits")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X = cd_edges[["circRNA", "disease"]]
y = cd_edges["label"]

Creating 5-fold splits


In [32]:
print("Building circRNA–disease graphs")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    print(f"Processing fold {fold}")

    train_edges = cd_edges.iloc[train_idx]
    train_pos = train_edges[train_edges["label"] == 1]

    train_edges.to_csv(
        DATA / f"circRNA_disease_fold{fold}_train.csv",
        index=False
    )
    
    test_edges = cd_edges.iloc[test_idx]
    
    test_edges.to_csv(
        DATA / f"circRNA_disease_fold{fold}_test.csv",
        index=False
    )

    cd_src = torch.tensor(
        train_pos["circRNA"].map(circ_map).values,
        dtype=torch.long
    )

    cd_dst = torch.tensor(
        train_pos["disease"].map(dis_map).values,
        dtype=torch.long
    )

    cd_edge = build_edge_index(
        train_pos,
        circ_map,
        dis_map,
        "circRNA",
        "disease"
    )

    # SHIFT disease node indices
    dis_offset = n_circ
    cd_edge[1] += dis_offset

    circ_cd_gip, _, dis_cd_gip, _ = compute_gip_edges(
        cd_src,
        cd_dst,
        n_circ,
        n_dis,
        k=10
    )

    # SHIFT disease similarity edges
    dis_cd_gip += dis_offset

    edge_index_cd = torch.cat([
        cd_edge,
        circ_cd_gip,
        dis_cd_gip
    ], dim=1)

    edge_index_cd = to_undirected(edge_index_cd)

    # ------------------------
    # Attach features
    # ------------------------

    x_cd = []

    circ_nodes = sorted(circ_map.keys(), key=lambda x: circ_map[x])
    dis_nodes = sorted(dis_map.keys(), key=lambda x: dis_map[x])

    for node in circ_nodes:
        x_cd.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

    for node in dis_nodes:
        x_cd.append(features.get(normalize_node(node), [0,0,0,0,0,0]))

    x_cd = torch.tensor(x_cd, dtype=torch.float)

    gcd = Data()
    gcd.edge_index = edge_index_cd
    gcd.x = x_cd
    gcd.num_circ = n_circ
    gcd.num_dis = n_dis

    torch.save(gcd, GRAPH_DIR / f"gcd_graph_fold{fold}.pt")

    print(f"Saved gcd_graph_fold{fold}.pt")
    print("Feature shape:", gcd.x.shape)

Building circRNA–disease graphs
Processing fold 0
Saved gcd_graph_fold0.pt
Feature shape: torch.Size([950, 6])
Processing fold 1
Saved gcd_graph_fold1.pt
Feature shape: torch.Size([950, 6])
Processing fold 2
Saved gcd_graph_fold2.pt
Feature shape: torch.Size([950, 6])
Processing fold 3
Saved gcd_graph_fold3.pt
Feature shape: torch.Size([950, 6])
Processing fold 4
Saved gcd_graph_fold4.pt
Feature shape: torch.Size([950, 6])


In [33]:
print("All graphs built successfully")

All graphs built successfully


In [34]:
import torch

gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)
gmd = torch.load(GRAPH_DIR / "gmd_graph.pt", weights_only=False)
gcd0 = torch.load(GRAPH_DIR / "gcd_graph_fold0.pt", weights_only=False)

print(gcm)
print("Edges:", gcm.edge_index.shape)
print(gmd)
print("Edges:", gmd.edge_index.shape)
print(gcd0)
print("Edges:", gcd0.edge_index.shape)

gcd0 = torch.load(GRAPH_DIR / "gcd_graph_fold0.pt", weights_only=False)
gcd1 = torch.load(GRAPH_DIR / "gcd_graph_fold1.pt", weights_only=False)
print(gcd0.edge_index.shape)
print(gcd1.edge_index.shape)

Data(edge_index=[2, 30196], x=[1510, 6], num_circ=828, num_mir=682)
Edges: torch.Size([2, 30196])
Data(edge_index=[2, 15316], x=[804, 6], num_mir=682, num_dis=122)
Edges: torch.Size([2, 15316])
Data(edge_index=[2, 17286], x=[950, 6], num_circ=828, num_dis=122)
Edges: torch.Size([2, 17286])
torch.Size([2, 17286])
torch.Size([2, 17484])


In [35]:
edges = gcm.edge_index

print("circ-circ edges:",
      ((edges[0] < 828) & (edges[1] < 828)).sum())

print("circ-mir edges:",
      ((edges[0] < 828) & (edges[1] >= 828)).sum())

print("mir-circ edges:",
      ((edges[0] >= 828) & (edges[1] < 828)).sum())

print("mir-mir edges:",
      ((edges[0] >= 828) & (edges[1] >= 828)).sum())

circ-circ edges: tensor(15488)
circ-mir edges: tensor(896)
mir-circ edges: tensor(896)
mir-mir edges: tensor(12916)


In [36]:
print("max node index:", gcm.edge_index.max())
print("expected max:", n_circ + n_mir - 1)

max node index: tensor(1509)
expected max: 1509


In [37]:
edges = gmd.edge_index

src = edges[0]
dst = edges[1]

mir_src = src < n_mir
mir_dst = dst < n_mir

dis_src = src >= n_mir
dis_dst = dst >= n_mir

print("mir-mir edges:", (mir_src & mir_dst).sum())
print("mir-dis edges:", (mir_src & dis_dst).sum())
print("dis-mir edges:", (dis_src & mir_dst).sum())
print("dis-dis edges:", (dis_src & dis_dst).sum())

mir-mir edges: tensor(11332)
mir-dis edges: tensor(828)
dis-mir edges: tensor(828)
dis-dis edges: tensor(2328)


In [38]:
print("max node index:", edges.max())
print("expected max:", 828 + 682 - 1)

max node index: tensor(803)
expected max: 1509


In [39]:
edges = gcd0.edge_index

print("circ-circ edges:",
      ((edges[0] < 828) & (edges[1] < 828)).sum())

print("circ-dis edges:",
      ((edges[0] < 828) & (edges[1] >= 828)).sum())

print("dis-circ edges:",
      ((edges[0] >= 828) & (edges[1] < 828)).sum())

print("dis-dis edges:",
      ((edges[0] >= 828) & (edges[1] >= 828)).sum())

circ-circ edges: tensor(13382)
circ-dis edges: tensor(788)
dis-circ edges: tensor(788)
dis-dis edges: tensor(2328)


In [40]:
edges = gcd0.edge_index

print("max node index:", edges.max())
print("expected max:", n_circ + n_dis - 1)

max node index: tensor(949)
expected max: 949


In [41]:
gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)
gmd = torch.load(GRAPH_DIR / "gmd_graph.pt", weights_only=False)
gcd0 = torch.load(GRAPH_DIR / "gcd_graph_fold0.pt", weights_only=False)

print("Gcm nodes:", gcm.edge_index.max()+1)
print("Gmd nodes:", gmd.edge_index.max()+1)
print("Gcd nodes:", gcd0.edge_index.max()+1)

print("Expected nodes Gcm:", n_circ + n_mir)
print("Expected nodes Gmd:", n_mir + n_dis)
print("Expected nodes Gcd:", n_circ + n_dis)

Gcm nodes: tensor(1510)
Gmd nodes: tensor(804)
Gcd nodes: tensor(950)
Expected nodes Gcm: 1510
Expected nodes Gmd: 804
Expected nodes Gcd: 950


In [42]:
# circRNA indices in Gcm
circ_nodes_gcm = gcm.edge_index[gcm.edge_index < n_circ]

# circRNA indices in Gcd
circ_nodes_gcd = gcd0.edge_index[gcd0.edge_index < n_circ]

print("Max circRNA index in Gcm:", circ_nodes_gcm.max())
print("Max circRNA index in Gcd:", circ_nodes_gcd.max())

Max circRNA index in Gcm: tensor(827)
Max circRNA index in Gcd: tensor(827)


In [43]:
mir_nodes_gcm = gcm.edge_index[
    (gcm.edge_index >= n_circ) &
    (gcm.edge_index < n_circ + n_mir)
]

mir_nodes_gmd = gmd.edge_index[gmd.edge_index < n_mir]

print("miRNA range in Gcm:", mir_nodes_gcm.min(), mir_nodes_gcm.max())
print("miRNA range in Gmd:", mir_nodes_gmd.min(), mir_nodes_gmd.max())

miRNA range in Gcm: tensor(828) tensor(1509)
miRNA range in Gmd: tensor(0) tensor(681)


In [44]:
print("Max node Gcm:", gcm.edge_index.max())
print("Expected:", n_circ + n_mir - 1)

print("Max node Gmd:", gmd.edge_index.max())
print("Expected:", n_mir + n_dis - 1)

print("Max node Gcd:", gcd0.edge_index.max())
print("Expected:", n_circ + n_dis - 1)

Max node Gcm: tensor(1509)
Expected: 1509
Max node Gmd: tensor(803)
Expected: 803
Max node Gcd: tensor(949)
Expected: 949


In [45]:
gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)

print(gcm)
print(gcm.x.shape)

Data(edge_index=[2, 30196], x=[1510, 6], num_circ=828, num_mir=682)
torch.Size([1510, 6])


In [46]:
missing = [node for node in mir_nodes if normalize_node(node) not in features]

print("Missing nodes:", len(missing))
print(missing[:10])

missing_set = set(missing)

used_missing_cm = cm_edges[
    cm_edges["miRNA"].apply(lambda x: normalize_node(x) in missing_set)
]

used_missing_md = md_edges[
    md_edges["miRNA"].apply(lambda x: normalize_node(x) in missing_set)
]

print("Missing nodes in CM edges:", len(used_missing_cm))
print("Missing nodes in MD edges:", len(used_missing_md))

Missing nodes: 161
['mir-490', 'mirna-218-5p', 'mir-125-5p', 'mir-372', 'mir-141', 'let-7b', 'mir-342-3p', 'mir-194-5p', 'mir-200s', 'mir-139']
Missing nodes in CM edges: 0
Missing nodes in MD edges: 0


In [47]:
gcm = torch.load(GRAPH_DIR / "gcm_graph.pt", weights_only=False)
gmd = torch.load(GRAPH_DIR / "gmd_graph.pt", weights_only=False)
gcd0 = torch.load(GRAPH_DIR / "gcd_graph_fold0.pt", weights_only=False)

print(gcm.x.shape)
print(gmd.x.shape)
print(gcd0.x.shape)

torch.Size([1510, 6])
torch.Size([804, 6])
torch.Size([950, 6])
